# Blizzard station ambience: make the 3-hour video
Snowed in at the mountain station: the crackling fire in front, the blizzard muffled behind the walls.

**Press the play button on the cell below**, allow Google Drive access when asked, and leave this tab open. It takes about 40 to 60 minutes and saves the finished video (about 9 GB) to **My Drive / Cozy Scenes**. Every step prints a line with the time, so you can see it working; if anything goes wrong it stops with a message saying what and why.

Nothing needs to be set.

In [ ]:
# Blizzard station ambience, 3 hours. Nothing to set: press the play button on the left.
SCENE, TYPE = "stationmaster", "ambience"
DRIVE_FOLDER = "Cozy Scenes"     # the finished video goes to My Drive / Cozy Scenes

import glob, json, os, shutil, subprocess, sys, threading, time, urllib.request

BRANCH = "claude/inspiring-faraday-20xabu"
CONTENT = os.environ.get("NB_CONTENT", "/content")          # Colab's working folder (changed only when testing elsewhere)
t0 = time.time()


def say(msg=""):
    print(f"[{(time.time() - t0) / 60:5.1f} min] {msg}", flush=True)


def stop(msg):
    print("\n" + "=" * 70 + "\nSTOPPED: " + msg + "\n" + "=" * 70, flush=True)
    raise SystemExit(1)


# 1. Google Drive ------------------------------------------------------------------------------------------------
say("Connecting Google Drive (allow access in the window that opens)...")
try:
    from google.colab import drive
    drive.mount(f"{CONTENT}/drive", force_remount=True)
    MYDRIVE = f"{CONTENT}/drive/MyDrive"
except ImportError:                                           # only when testing outside Colab
    MYDRIVE = os.environ["NB_DRIVE"]
    drive = None
OUT_DIR = f"{MYDRIVE}/{DRIVE_FOLDER}"
os.makedirs(OUT_DIR, exist_ok=True)
say(f"Drive connected. The finished video will be saved in My Drive / {DRIVE_FOLDER}")

# 2. Room to work ------------------------------------------------------------------------------------------------
free = shutil.disk_usage(CONTENT).free / 1e9
say(f"Free space on this Colab machine: {free:.0f} GB")
if free < 25:
    stop(f"Only {free:.0f} GB free here; about 25 GB is needed. Runtime > Disconnect and delete runtime, then run again.")
ff = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if ff.returncode:
    stop("ffmpeg is missing on this machine. Runtime > Disconnect and delete runtime, then run again.")
say("Video tool: " + ff.stdout.splitlines()[0])

# 3. The kit: picture, sound and build script from GitHub ----------------------------------------------------------
BASE = f"https://raw.githubusercontent.com/codingbythy/claude/refs/heads/{BRANCH}/fox-scenes/{SCENE}/video-kit/"
work = f"{CONTENT}/{SCENE}-{TYPE}"
os.makedirs(work, exist_ok=True)
os.chdir(work)


def fetch(name, tries=4):
    for k in range(tries):
        try:
            with urllib.request.urlopen(BASE + name, timeout=120) as r:
                size = int(r.headers.get("Content-Length") or 0); data = r.read()
            if size and len(data) != size:
                raise IOError(f"got {len(data)} of {size} bytes")
            with open(name, "wb") as f:
                f.write(data)
            return len(data)
        except Exception as e:
            if k == tries - 1:
                stop(f"Could not download {name} from GitHub ({e}). Check the internet connection and run again.")
            say(f"  {name}: {e}; trying again"); time.sleep(5 * (k + 1))


say("Downloading the kit from GitHub...")
fetch("kit.json")
kit = json.load(open("kit.json"))
if TYPE not in kit["mixes"]:
    stop(f"This scene has no {TYPE} version; it has {', '.join(kit['mixes'])}.")
for name in kit["files"]:
    say(f"  {name}: {fetch(name) / 1e6:.1f} MB")
hours = kit["mixes"][TYPE].get("hours", kit["hours"])
out = f"{kit['slug']}-{TYPE}-{hours}h.mp4"

# 4. The fire's sound files, from your Drive (myNoise, used with permission; never in the public repository) -------------
extra = []
if kit.get("fire"):
    F = kit["fire"]; need = [band[s] + ".ogg" for band in F["bands"] for s in ("a", "b")]
    say(f"Looking for the fire's sound files ({F['drive_folder']}) in your Drive...")
    places = [f"{OUT_DIR}/{F['drive_folder']}"] + glob.glob(f"{MYDRIVE}/*/{F['drive_folder']}") + glob.glob(f"{MYDRIVE}/*/*/{F['drive_folder']}") \
        + glob.glob(f"{CONTENT}/drive/Shareddrives/*/{F['drive_folder']}") + glob.glob(f"{CONTENT}/drive/Shareddrives/*/*/{F['drive_folder']}")
    found = next((p for p in places if all(os.path.exists(f"{p}/{n}") for n in need)), None)
    local = f"{CONTENT}/fire-{SCENE}"; os.makedirs(local, exist_ok=True)
    if found:
        for n in need:
            shutil.copy(f"{found}/{n}", local)
        say(f"  found in {found.replace(CONTENT + '/drive/', '')}")
    else:                         # not in Drive: fetch the same files the myNoise website plays (used with the owner's permission)
        say("  not in your Drive; downloading them from myNoise instead (used with permission)")
        for n in need:
            for k in range(4):
                try:
                    req = urllib.request.Request(f"https://mynoise.world/Data/FIRE/{n}", headers={"User-Agent": "Mozilla/5.0", "Referer": "https://mynoise.net/NoiseMachines/fireNoiseGenerator.php"})
                    with urllib.request.urlopen(req, timeout=60) as r, open(f"{local}/{n}", "wb") as f:
                        f.write(r.read())
                    break
                except Exception as e:
                    if k == 3:
                        stop(f"The fire's sound file {n} is not in My Drive / {DRIVE_FOLDER} / {F['drive_folder']} and could not be downloaded ({e}).")
                    time.sleep(5)
    extra = ["--fire-dir", local]
    say(f"Fire: {F['name']}")

# 5. Build --------------------------------------------------------------------------------------------------------------
say(f"Building {out} ({hours} hours). This is the long part: about 30 to 50 minutes. Keep this tab open.")
cmd = [sys.executable, "-u", "build.py", out, "--type", TYPE] + extra
if os.environ.get("NB_TEST_LOOPS"):
    cmd += ["--loops", os.environ["NB_TEST_LOOPS"]]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
last = [time.time()]


def heartbeat():
    while p.poll() is None:
        time.sleep(30)
        if p.poll() is None and time.time() - last[0] > 120:
            say("  still working..."); last[0] = time.time()


threading.Thread(target=heartbeat, daemon=True).start()
lines = []
for line in p.stdout:
    line = line.rstrip(); lines.append(line); last[0] = time.time(); say(line)
if p.wait() != 0:
    stop("The build failed. The lines just above say why; send them to Claude.")
if not os.path.exists(out) or os.path.getsize(out) < 1e6:
    stop("The build finished but the video file is missing or empty.")
dur = float(subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=nw=1:nk=1", out], capture_output=True, text=True).stdout or 0)
size = os.path.getsize(out) / 1e9
say(f"Video made: {int(dur // 3600)} h {int(dur % 3600 // 60)} min {dur % 60:.0f} s, {size:.1f} GB")

# 6. Save to Drive ------------------------------------------------------------------------------------------------------
st = os.statvfs(OUT_DIR); room = st.f_bavail * st.f_frsize / 1e9
if room and room < size + 0.5:
    stop(f"Your Google Drive has {room:.1f} GB free but the video is {size:.1f} GB. Free up space, then run the cell again.")
say(f"Copying to My Drive / {DRIVE_FOLDER} ({size:.1f} GB, a few minutes)...")
shutil.copy(out, f"{OUT_DIR}/{out}")
if os.path.getsize(f"{OUT_DIR}/{out}") != os.path.getsize(out):
    stop("The copy to Drive is incomplete. Run the cell again.")
if drive:
    drive.flush_and_unmount()
say(f"DONE: My Drive / {DRIVE_FOLDER} / {out}")
say("Upload it with the title, description and thumbnail from the scene's youtube folder.")
